# Unemployment Analysis with Python – India

## Objective
Explore regional and temporal unemployment trends in India, with a focus on the COVID-19 period.

**Dataset:** Kaggle – [Unemployment in India](https://www.kaggle.com/datasets/gokulrajkmv/unemployment-in-india)

**Important data note:** The commonly used Kaggle file contains `Estimated Employed` (an estimated count), not an employment *rate*. Therefore, the correlation analysis uses the employed count and labels it accurately. An employment-to-population rate cannot be calculated without a suitable working-age population denominator.

## 1. Install / import libraries

If any library is missing, run `pip install pandas numpy matplotlib seaborn jupyter` in your terminal.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)

## 2. Download and load the dataset

1. Open the [Kaggle dataset page](https://www.kaggle.com/datasets/gokulrajkmv/unemployment-in-india).
2. Download and extract the ZIP.
3. Copy **`Unemployment in India.csv`** into this project's `data` folder.

The notebook also checks the current folder if the file is stored beside the notebook.

In [ ]:
candidate_paths = [
    Path("data/Unemployment in India.csv"),
    Path("Unemployment in India.csv"),
    Path("data/Unemployment_in_India.csv"),
    Path("Unemployment_in_India.csv")
]

data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Dataset CSV not found. Download 'Unemployment in India.csv' from "
        "https://www.kaggle.com/datasets/gokulrajkmv/unemployment-in-india "
        "and place it in the project's data/ folder."
    )

df = pd.read_csv(data_path)
print("Loaded:", data_path)
print("Dataset shape:", df.shape)
display(df.head())

## 3. Inspect and clean the data

In [ ]:
# Clean accidental spaces in column names and text fields
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("string").str.strip()

print("Column names:", df.columns.tolist())
print("\\nData types before conversion:")
print(df.dtypes)
print("\\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Convert date strings such as 31-05-2019 to datetime
if "Date" not in df.columns:
    raise KeyError("Expected a 'Date' column in the downloaded dataset.")

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

# Convert expected numeric columns safely
numeric_candidates = [
    "Estimated Unemployment Rate (%)",
    "Estimated Employed",
    "Estimated Labour Participation Rate (%)"
]

for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove rows without a usable date or region
if "Region" not in df.columns:
    raise KeyError("Expected a 'Region' column in the downloaded dataset.")

df = df.dropna(subset=["Date", "Region"]).copy()
df = df.drop_duplicates()

print("Shape after basic cleaning:", df.shape)
print("\\nMissing values after cleaning:")
print(df.isnull().sum())
display(df.describe(include="all").T)

## 4. Add time features

We create a calendar month field for month-wise analysis. The COVID period is defined here as **1 March 2020 onward**, a practical analytical cutoff that includes the nationwide lockdown period. This is a descriptive comparison, not proof that COVID alone caused every change.

In [ ]:
unemployment_col = "Estimated Unemployment Rate (%)"
employed_col = "Estimated Employed"
labour_col = "Estimated Labour Participation Rate (%)"

required = [unemployment_col, employed_col, labour_col]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise KeyError(f"Dataset is missing expected columns: {missing_columns}")

df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.month_name()
df["Year_Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
df["Period"] = np.where(df["Date"] < pd.Timestamp("2020-03-01"),
                        "Pre-COVID", "COVID period (Mar 2020 onward)")

display(df[["Region", "Date", "Year_Month", unemployment_col, "Period"]].head())

## 5. Region-wise average unemployment rate

In [ ]:
region_avg = (
    df.groupby("Region")[unemployment_col]
      .mean()
      .sort_values(ascending=False)
)

display(region_avg.to_frame("Average unemployment rate (%)").round(2))

plt.figure(figsize=(12, 8))
sns.barplot(x=region_avg.values, y=region_avg.index, color="steelblue")
plt.title("Average Unemployment Rate by State / Region")
plt.xlabel("Average unemployment rate (%)")
plt.ylabel("State / Region")
plt.tight_layout()
plt.show()

## 6. Month-wise unemployment trend

In [ ]:
monthly_avg = (
    df.groupby("Year_Month")[unemployment_col]
      .mean()
      .sort_index()
)

display(monthly_avg.to_frame("Mean unemployment rate (%)").round(2))

plt.figure(figsize=(12, 5))
plt.plot(monthly_avg.index, monthly_avg.values, marker="o")
plt.title("Month-wise Average Unemployment Rate in India (Dataset Sample)")
plt.xlabel("Month")
plt.ylabel("Average unemployment rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. State-wise time-series line chart (at least 3 regions)

In [ ]:
# Select three regions with the most observations to ensure usable time series
region_counts = df.groupby("Region")["Date"].nunique().sort_values(ascending=False)
selected_regions = region_counts.head(3).index.tolist()
print("Regions plotted:", selected_regions)

state_monthly = (
    df[df["Region"].isin(selected_regions)]
      .groupby(["Year_Month", "Region"])[unemployment_col]
      .mean()
      .reset_index()
)

plt.figure(figsize=(13, 6))
sns.lineplot(data=state_monthly, x="Year_Month", y=unemployment_col,
             hue="Region", marker="o")
plt.title("Unemployment Rate Over Time – Three Regions")
plt.xlabel("Month")
plt.ylabel("Unemployment rate (%)")
plt.xticks(rotation=45)
plt.legend(title="Region")
plt.tight_layout()
plt.show()

## 8. Top 10 regions with highest average unemployment

In [ ]:
top10 = region_avg.head(10).sort_values()

plt.figure(figsize=(10, 6))
sns.barplot(x=top10.values, y=top10.index, color="indianred")
plt.title("Top 10 States / Regions by Average Unemployment Rate")
plt.xlabel("Average unemployment rate (%)")
plt.ylabel("State / Region")
plt.tight_layout()
plt.show()

display(region_avg.head(10).to_frame("Average unemployment rate (%)").round(2))

## 9. Correlation heatmap

In [ ]:
# Note: Estimated Employed is a count, not an employment rate.
corr_cols = [unemployment_col, employed_col, labour_col]
corr = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f",
            vmin=-1, vmax=1)
plt.title("Correlation: Unemployment, Estimated Employed Count, Labour Participation")
plt.tight_layout()
plt.show()

display(corr.round(3))

## 10. Pre-COVID vs COVID-period comparison

In [ ]:
period_summary = (
    df.groupby("Period")[[unemployment_col, employed_col, labour_col]]
      .mean()
      .reindex(["Pre-COVID", "COVID period (Mar 2020 onward)"])
)

display(period_summary.round(2))

plt.figure(figsize=(9, 5))
sns.barplot(
    data=df, x="Period", y=unemployment_col,
    order=["Pre-COVID", "COVID period (Mar 2020 onward)"],
    estimator=np.mean, errorbar=None, color="slateblue"
)
plt.title("Average Unemployment Rate: Pre-COVID vs COVID Period")
plt.xlabel("Period")
plt.ylabel("Mean unemployment rate (%)")
plt.tight_layout()
plt.show()

## 11. Month-of-year pattern

In [ ]:
month_order = list(range(1, 13))
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

month_pattern = df.groupby("Month")[unemployment_col].mean().reindex(month_order)

plt.figure(figsize=(10, 5))
sns.barplot(x=month_labels, y=month_pattern.values, color="teal")
plt.title("Average Unemployment Rate by Calendar Month")
plt.xlabel("Month")
plt.ylabel("Average unemployment rate (%)")
plt.tight_layout()
plt.show()

## 12. Final summary and conclusion

In [ ]:
pre = period_summary.loc["Pre-COVID", unemployment_col]
post = period_summary.loc["COVID period (Mar 2020 onward)", unemployment_col]
difference = post - pre

print("PROJECT SUMMARY")
print("Rows analyzed:", len(df))
print("Regions:", df["Region"].nunique())
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print(f"Pre-COVID mean unemployment: {pre:.2f}%")
print(f"COVID-period mean unemployment: {post:.2f}%")
print(f"Difference (COVID period - Pre-COVID): {difference:+.2f} percentage points")
print("Highest average region:", region_avg.index[0])
print(f"Highest region average: {region_avg.iloc[0]:.2f}%")